# LeetCode 1150 — Check If a Number Is Majority Element in a Sorted Array bằng Python

**Link đề:** https://leetcode.com/problems/check-if-a-number-is-majority-element-in-a-sorted-array/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def isMajorityElement(self, nums: List[int], target: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **majority element** (phần tử chiếm quá nửa) và điều kiện `count > n / 2` viết bằng số nguyên.
2. Giải baseline bằng **đếm tuyến tính** `O(n)`.
3. Khai thác dữ kiện **mảng đã sắp xếp**: các phần tử bằng `target` nằm **liền một khối**.
4. Học **binary search** tìm vị trí đầu/cuối của khối (tự viết và dùng `bisect`).
5. Mẹo "nhảy nửa mảng": chỉ cần **một** lần binary search rồi kiểm tra `nums[i + n // 2]`.
6. Trace, chứng minh, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `nums` **đã sắp xếp không giảm** và một số `target`. Trả về `True` nếu `target` là **majority element** của `nums`, ngược lại `False`.

**Majority element** là phần tử xuất hiện **nhiều hơn `nums.length / 2` lần**.

| `nums` | `target` | Output | Giải thích |
|---|:---:|:---:|---|
| `[2,4,5,5,5,5,5,6,6]` | `5` | `True` | 5 xuất hiện 5 lần, `5 > 9/2 = 4.5` |
| `[10,100,101,101]` | `101` | `False` | 101 xuất hiện 2 lần, `2 > 4/2 = 2` là sai |

**Constraints (theo hiểu biết):**
- `1 <= nums.length <= 1000`
- `1 <= nums[i], target <= 10^9`
- `nums` được sắp xếp không giảm.

In [1]:
from typing import List   # LeetCode dùng List[int] trong chữ ký hàm

nums = [2, 4, 5, 5, 5, 5, 5, 6, 6]
target = 5
print("n =", len(nums), "| số lần target xuất hiện =", nums.count(target), "| n/2 =", len(nums) / 2)

n = 9 | số lần target xuất hiện = 5 | n/2 = 4.5


### "Nhiều hơn một nửa" — cẩn thận chữ **hơn**

- `n = 4`: cần **ít nhất 3** lần (2 lần chỉ bằng một nửa, không đủ).
- `n = 9`: cần **ít nhất 5** lần.

Viết bằng số nguyên để khỏi dùng số thực: `count > n / 2` ⇔ `count * 2 > n` ⇔ `count > n // 2` ⇔ `count >= n // 2 + 1`.

In [2]:
for n in [1, 2, 3, 4, 5, 9, 10]:
    need = n // 2 + 1
    print(f"n={n:>2}: cần ít nhất {need} lần  (kiểm tra: {need}*2 > {n} là {need*2 > n}, {need-1}*2 > {n} là {(need-1)*2 > n})")

n= 1: cần ít nhất 1 lần  (kiểm tra: 1*2 > 1 là True, 0*2 > 1 là False)
n= 2: cần ít nhất 2 lần  (kiểm tra: 2*2 > 2 là True, 1*2 > 2 là False)
n= 3: cần ít nhất 2 lần  (kiểm tra: 2*2 > 3 là True, 1*2 > 3 là False)
n= 4: cần ít nhất 3 lần  (kiểm tra: 3*2 > 4 là True, 2*2 > 4 là False)
n= 5: cần ít nhất 3 lần  (kiểm tra: 3*2 > 5 là True, 2*2 > 5 là False)
n= 9: cần ít nhất 5 lần  (kiểm tra: 5*2 > 9 là True, 4*2 > 9 là False)
n=10: cần ít nhất 6 lần  (kiểm tra: 6*2 > 10 là True, 5*2 > 10 là False)


## 2. Nền tảng Python cần biết

### 2.1. `//` (chia lấy phần nguyên) khác `/`
- `9 / 2 = 4.5` (số thực)
- `9 // 2 = 4` (bỏ phần lẻ)

In [3]:
print(9 / 2, 9 // 2)
print(4 / 2, 4 // 2)

4.5 4
2.0 2


### 2.2. `list.count(x)` và đếm bằng vòng lặp

`nums.count(x)` duyệt cả list và trả số lần `x` xuất hiện — tốn `O(n)`.

In [4]:
nums = [10, 100, 101, 101]
print(nums.count(101))
cnt = 0
for x in nums:
    if x == 101:
        cnt += 1
print(cnt)

2
2


### 2.3. Mảng đã sắp xếp → các giá trị bằng nhau đứng cạnh nhau

Đây là chìa khóa. Trong `[2,4,5,5,5,5,5,6,6]`, các số `5` chiếm **đoạn liên tiếp** từ index 2 tới index 6. Nếu biết **đầu** và **cuối** đoạn, số lần xuất hiện = `cuối - đầu + 1` — không cần duyệt hết.

In [5]:
nums = [2, 4, 5, 5, 5, 5, 5, 6, 6]
for i, x in enumerate(nums):
    print(i, x, "<- target" if x == 5 else "")

0 2 
1 4 
2 5 <- target
3 5 <- target
4 5 <- target
5 5 <- target
6 5 <- target
7 6 
8 6 


`enumerate(nums)` trả về từng cặp `(index, giá trị)` — tiện khi cần cả vị trí lẫn giá trị.

### 2.4. Binary search (tìm kiếm nhị phân) — ý tưởng

Trên mảng đã sắp xếp, muốn tìm vị trí đầu tiên có giá trị `>= target`:
- Xét phần tử giữa `nums[mid]`.
- Nếu `nums[mid] < target` → đáp án nằm **bên phải** mid.
- Ngược lại → đáp án là mid hoặc **bên trái** mid.

Mỗi bước loại một nửa → chỉ cần khoảng `log2(n)` bước. Với `n = 1000` là ~10 bước.

In [6]:
import math
for n in [8, 1000, 10**6]:
    print(f"n={n:>8}: log2(n) ≈ {math.log2(n):.1f} bước")

n=       8: log2(n) ≈ 3.0 bước
n=    1000: log2(n) ≈ 10.0 bước
n= 1000000: log2(n) ≈ 19.9 bước


### 2.5. Module `bisect` — binary search có sẵn

- `bisect_left(a, x)`: vị trí **đầu tiên** mà `a[i] >= x` (chỗ chèn `x` bên trái các bản sao).
- `bisect_right(a, x)`: vị trí **đầu tiên** mà `a[i] > x` (chỗ chèn bên phải các bản sao).
- Số lần `x` xuất hiện = `bisect_right - bisect_left`.

In [7]:
from bisect import bisect_left, bisect_right

a = [2, 4, 5, 5, 5, 5, 5, 6, 6]
print("bisect_left(5) =", bisect_left(a, 5))    # 2
print("bisect_right(5) =", bisect_right(a, 5))  # 7
print("số lần 5 =", bisect_right(a, 5) - bisect_left(a, 5))
print("bisect_left(3) =", bisect_left(a, 3), "| bisect_right(3) =", bisect_right(a, 3), "-> 3 không có")

bisect_left(5) = 2
bisect_right(5) = 7
số lần 5 = 5
bisect_left(3) = 1 | bisect_right(3) = 1 -> 3 không có


## 3. Cách 1 — Baseline: đếm tuyến tính

Đếm số lần `target` xuất hiện rồi so với `n // 2`. Không dùng tới tính chất đã sắp xếp.

In [8]:
def is_majority_count(nums: List[int], target: int) -> bool:
    count = 0
    for x in nums:
        if x == target:
            count += 1
    return count > len(nums) // 2

print(is_majority_count([2, 4, 5, 5, 5, 5, 5, 6, 6], 5))   # True
print(is_majority_count([10, 100, 101, 101], 101))         # False

True
False


Viết gọn: `return nums.count(target) > len(nums) // 2`. Time `O(n)`, Space `O(1)`.

**Cải tiến nhỏ — dừng sớm:** vì mảng sắp xếp, khi đã gặp số `> target` thì phía sau không còn `target` nữa → `break`. Vẫn `O(n)` trường hợp xấu nhất.

In [9]:
def is_majority_early_stop(nums: List[int], target: int) -> bool:
    count = 0
    for x in nums:
        if x > target:
            break           # mảng tăng dần: phía sau chỉ còn số lớn hơn
        if x == target:
            count += 1
    return count > len(nums) // 2

print(is_majority_early_stop([2, 4, 5, 5, 5, 5, 5, 6, 6], 5))
print(is_majority_early_stop([10, 100, 101, 101], 101))

True
False


## 4. Nút thắt và insight

**Nút thắt:** baseline nhìn **từng phần tử** — lãng phí khi đề đã cho mảng **sắp xếp**.

**Insight 1:** các `target` nằm liền một khối `[first, last]`. Tìm `first` và `last` bằng binary search → `O(log n)`.

**Insight 2 (gọn hơn):** nếu `target` chiếm **quá nửa** và khối bắt đầu ở `first`, thì khối phải kéo dài **ít nhất tới `first + n // 2`**. Vậy chỉ cần kiểm tra:

```text
first + n // 2 < n   và   nums[first + n // 2] == target
```

Ví dụ `[2,4,5,5,5,5,5,6,6]`, `n = 9`, `first = 2`, `n // 2 = 4` → xem `nums[6] = 5` ✔ → True.

In [10]:
nums = [2, 4, 5, 5, 5, 5, 5, 6, 6]
n = len(nums)
first = bisect_left(nums, 5)
j = first + n // 2
print("first =", first, "| j = first + n//2 =", j, "| nums[j] =", nums[j])

nums2 = [10, 100, 101, 101]
first2 = bisect_left(nums2, 101)
j2 = first2 + len(nums2) // 2
print("first =", first2, "| j =", j2, "| j < n?", j2 < len(nums2))

first = 2 | j = first + n//2 = 6 | nums[j] = 5
first = 2 | j = 4 | j < n? False


**Vì sao insight 2 đúng?** Khối các `target` là đoạn `[first, last]`, độ dài `count = last - first + 1`.
- `count > n // 2` ⇔ `count >= n // 2 + 1` ⇔ `last >= first + n // 2`.
- Do mảng sắp xếp và `nums[first] == target`, điều kiện `last >= j` (với `j = first + n // 2`) tương đương `nums[j] == target` (mọi ô từ `first` tới `last` đều là target, ô sau `last` thì lớn hơn).

## 5. Cách 2 — Tự viết binary search tìm `first`

Hàm `lower_bound(nums, target)` trả về index đầu tiên `i` có `nums[i] >= target` (trả `n` nếu không có).

**Mổ xẻ:**
1. `lo, hi = 0, n` — vùng tìm kiếm nửa mở `[lo, hi)`.
2. `while lo < hi`: còn ít nhất một ứng viên.
3. `mid = (lo + hi) // 2`.
4. `nums[mid] < target` → đáp án ở phải: `lo = mid + 1`.
5. Ngược lại → đáp án là `mid` hoặc ở trái: `hi = mid`.
6. Kết thúc `lo == hi` chính là đáp án.

In [11]:
def lower_bound(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [2, 4, 5, 5, 5, 5, 5, 6, 6]
for t in [1, 2, 5, 6, 7, 3]:
    print(f"lower_bound({t}) = {lower_bound(a, t)}  | bisect_left = {bisect_left(a, t)}")

lower_bound(1) = 0  | bisect_left = 0
lower_bound(2) = 0  | bisect_left = 0
lower_bound(5) = 2  | bisect_left = 2
lower_bound(6) = 7  | bisect_left = 7
lower_bound(7) = 9  | bisect_left = 9
lower_bound(3) = 1  | bisect_left = 1


### Ghép thành lời giải chính

In [12]:
def is_majority_binary(nums: List[int], target: int) -> bool:
    n = len(nums)
    first = lower_bound(nums, target)
    j = first + n // 2
    return j < n and nums[j] == target

print(is_majority_binary([2, 4, 5, 5, 5, 5, 5, 6, 6], 5))   # True
print(is_majority_binary([10, 100, 101, 101], 101))         # False

True
False


`j < n and nums[j] == target`: toán tử `and` **ngắt mạch** — nếu `j < n` sai thì Python không tính vế sau, nhờ đó không bị `IndexError`.

In [13]:
arr = [1, 2]
k = 5
print(k < len(arr) and arr[k] == 2)    # False, không lỗi
try:
    print(arr[k] == 2 and k < len(arr))
except IndexError as e:
    print("Đảo thứ tự → IndexError:", e)

False
Đảo thứ tự → IndexError: list index out of range


## 6. TRACE từng bước

Trace `lower_bound([2,4,5,5,5,5,5,6,6], 5)`:

| vòng | lo | hi | mid | nums[mid] | so với 5 | cập nhật |
|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 9 | 4 | 5 | không `<` | hi = 4 |
| 2 | 0 | 4 | 2 | 5 | không `<` | hi = 2 |
| 3 | 0 | 2 | 1 | 4 | `<` | lo = 2 |
| dừng | 2 | 2 | | | | first = 2 |

Sau đó `j = 2 + 9 // 2 = 6`, `nums[6] = 5` → **True**.

In [14]:
def trace_is_majority(nums: List[int], target: int) -> bool:
    n = len(nums)
    lo, hi = 0, n
    step = 1
    print(f"nums={nums}, target={target}, n={n}")
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            print(f"  vòng {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={nums[mid]} < {target} -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"  vòng {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={nums[mid]} >= {target} -> hi = {mid}")
            hi = mid
        step += 1
    first = lo
    j = first + n // 2
    print(f"  first = {first}, j = first + n//2 = {j}")
    if j >= n:
        print("  j vượt mảng -> False")
        return False
    ok = nums[j] == target
    print(f"  nums[{j}] = {nums[j]} -> {'== target -> True' if ok else '!= target -> False'}")
    return ok

trace_is_majority([2, 4, 5, 5, 5, 5, 5, 6, 6], 5)
print()
trace_is_majority([10, 100, 101, 101], 101)
print()
trace_is_majority([1, 1, 2, 2], 1)

nums=[2, 4, 5, 5, 5, 5, 5, 6, 6], target=5, n=9
  vòng 1: lo=0, hi=9, mid=4, nums[mid]=5 >= 5 -> hi = 4
  vòng 2: lo=0, hi=4, mid=2, nums[mid]=5 >= 5 -> hi = 2
  vòng 3: lo=0, hi=2, mid=1, nums[mid]=4 < 5 -> lo = 2
  first = 2, j = first + n//2 = 6
  nums[6] = 5 -> == target -> True

nums=[10, 100, 101, 101], target=101, n=4
  vòng 1: lo=0, hi=4, mid=2, nums[mid]=101 >= 101 -> hi = 2
  vòng 2: lo=0, hi=2, mid=1, nums[mid]=100 < 101 -> lo = 2
  first = 2, j = first + n//2 = 4
  j vượt mảng -> False

nums=[1, 1, 2, 2], target=1, n=4
  vòng 1: lo=0, hi=4, mid=2, nums[mid]=2 >= 1 -> hi = 2
  vòng 2: lo=0, hi=2, mid=1, nums[mid]=1 >= 1 -> hi = 1
  vòng 3: lo=0, hi=1, mid=0, nums[mid]=1 >= 1 -> hi = 0
  first = 0, j = first + n//2 = 2
  nums[2] = 2 -> != target -> False


False

Ca thứ 3 thú vị: `[1,1,2,2]`, target 1: `first = 0`, `j = 2`, `nums[2] = 2 ≠ 1` → False (1 xuất hiện đúng **một nửa**, không phải **hơn** nửa).

## 7. Cách 3 — Hai lần `bisect` để đếm chính xác

Dễ đọc nhất trong các cách `O(log n)`: đếm = `bisect_right - bisect_left`.

In [15]:
def is_majority_bisect(nums: List[int], target: int) -> bool:
    count = bisect_right(nums, target) - bisect_left(nums, target)
    return count > len(nums) // 2

print(is_majority_bisect([2, 4, 5, 5, 5, 5, 5, 6, 6], 5))
print(is_majority_bisect([10, 100, 101, 101], 101))

True
False


## 8. Code nộp LeetCode

### Bản chính — một lần `bisect_left` + nhảy nửa mảng

In [16]:
from bisect import bisect_left
from typing import List

class Solution:
    def isMajorityElement(self, nums: List[int], target: int) -> bool:
        n = len(nums)
        first = bisect_left(nums, target)
        j = first + n // 2
        return j < n and nums[j] == target

sol = Solution()
print(sol.isMajorityElement([2, 4, 5, 5, 5, 5, 5, 6, 6], 5))
print(sol.isMajorityElement([10, 100, 101, 101], 101))

True
False


### Bản đếm tuyến tính (cũng được chấp nhận vì `n <= 1000`)

In [17]:
class SolutionCount:
    def isMajorityElement(self, nums: List[int], target: int) -> bool:
        return nums.count(target) > len(nums) // 2

class SolutionTwoBisect:
    def isMajorityElement(self, nums: List[int], target: int) -> bool:
        return bisect_right(nums, target) - bisect_left(nums, target) > len(nums) // 2

print(SolutionCount().isMajorityElement([2, 4, 5, 5, 5, 5, 5, 6, 6], 5),
      SolutionTwoBisect().isMajorityElement([10, 100, 101, 101], 101))

True False


## 9. Vì sao đúng? (invariant của binary search)

**Invariant của `lower_bound`:** trong suốt vòng lặp,
- mọi index `< lo` có `nums[i] < target`;
- mọi index `>= hi` có `nums[i] >= target`.

Ban đầu `lo = 0, hi = n` — đúng vì hai vùng đều rỗng. Mỗi nhánh cập nhật giữ invariant (vì mảng sắp xếp). Vùng `[lo, hi)` co lại mỗi vòng nên vòng lặp dừng, khi đó `lo == hi` là index đầu tiên `>= target`.

**Bước kiểm tra:** như mục 4 đã chứng minh, `count > n // 2` ⇔ `nums[first + n // 2] == target` (với điều kiện index hợp lệ). Nếu `target` không có trong mảng, `nums[first]` (nếu tồn tại) `> target`, nên `nums[j] >= nums[first] > target` → trả False đúng.

## 10. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| Đếm tuyến tính / `count` | `O(n)` | `O(1)` |
| Đếm có dừng sớm | `O(n)` xấu nhất | `O(1)` |
| Hai lần bisect | `O(log n)` | `O(1)` |
| Một bisect + nhảy nửa | `O(log n)` | `O(1)` |

Với `n <= 1000` khác biệt nhỏ, nhưng bài này dạy đúng một thói quen: **thấy "đã sắp xếp" → nghĩ binary search**.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

- Mẫu đề.
- `n = 1`: phần tử duy nhất bằng / khác target.
- Đúng một nửa (`[1,1,2,2]`, target 1) → False.
- Target không có trong mảng, nhỏ hơn mọi phần tử, lớn hơn mọi phần tử.
- Toàn bộ mảng là target.

In [18]:
cases = [
    ([2, 4, 5, 5, 5, 5, 5, 6, 6], 5, True),
    ([10, 100, 101, 101], 101, False),
    ([1], 1, True),
    ([1], 2, False),
    ([1, 1, 2, 2], 1, False),
    ([1, 1, 2, 2], 2, False),
    ([1, 1, 1, 2], 1, True),
    ([1, 2, 2, 2], 2, True),
    ([3, 3, 3], 3, True),
    ([1, 2, 3], 0, False),
    ([1, 2, 3], 9, False),
    ([1, 2, 3, 3, 3], 3, True),
    ([1, 1, 2, 3, 3], 3, False),
]
solvers = [
    ("count", is_majority_count),
    ("early_stop", is_majority_early_stop),
    ("binary", is_majority_binary),
    ("bisect2", is_majority_bisect),
    ("Solution", Solution().isMajorityElement),
]
for nums, t, expected in cases:
    for name, f in solvers:
        assert f(nums, t) == expected, (name, nums, t)
    print(f"{str(nums):<28} target={t:<3} -> {expected}  PASS")

[2, 4, 5, 5, 5, 5, 5, 6, 6]  target=5   -> True  PASS
[10, 100, 101, 101]          target=101 -> False  PASS
[1]                          target=1   -> True  PASS
[1]                          target=2   -> False  PASS
[1, 1, 2, 2]                 target=1   -> False  PASS
[1, 1, 2, 2]                 target=2   -> False  PASS
[1, 1, 1, 2]                 target=1   -> True  PASS
[1, 2, 2, 2]                 target=2   -> True  PASS
[3, 3, 3]                    target=3   -> True  PASS
[1, 2, 3]                    target=0   -> False  PASS
[1, 2, 3]                    target=9   -> False  PASS
[1, 2, 3, 3, 3]              target=3   -> True  PASS
[1, 1, 2, 3, 3]              target=3   -> False  PASS


In [19]:
import random
random.seed(1150)
for _ in range(2000):
    n = random.randint(1, 12)
    nums = sorted(random.randint(1, 4) for _ in range(n))
    t = random.randint(0, 5)
    expected = is_majority_count(nums, t)
    assert is_majority_binary(nums, t) == expected
    assert is_majority_bisect(nums, t) == expected
    assert Solution().isMajorityElement(nums, t) == expected
print("2000 test ngẫu nhiên (seed cố định) khớp baseline. PASS")

2000 test ngẫu nhiên (seed cố định) khớp baseline. PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì `>`
```python
return count >= len(nums) // 2     # [1,1,2,2], 1 → True (SAI)
```
"Majority" là **quá** nửa. Sửa: `count > len(nums) // 2`.

### Lỗi 2: Kiểm tra `nums[first + n // 2]` mà quên điều kiện `j < n` → `IndexError`.

### Lỗi 3: Dùng `first + n // 2 - 1` (lệch một)
Khi đó chỉ kiểm tra được `count >= n // 2`, tức "ít nhất một nửa" → sai với ca đúng một nửa.

### Lỗi 4: Quên kiểm tra `nums[first] == target`
Ở bản "nhảy nửa" không cần, vì `nums[j] == target` đã bao hàm. Nhưng ở bản tự viết `first`/`last` mà không kiểm tra thì có thể đếm sai khi target vắng mặt.

In [20]:
def wrong_ge(nums, target):
    return nums.count(target) >= len(nums) // 2

def wrong_minus_one(nums, target):
    n = len(nums)
    j = bisect_left(nums, target) + n // 2 - 1
    return 0 <= j < n and nums[j] == target

def wrong_no_bound(nums, target):
    j = bisect_left(nums, target) + len(nums) // 2
    return nums[j] == target

print("đúng:       ", Solution().isMajorityElement([1, 1, 2, 2], 1))
print("dùng >=:    ", wrong_ge([1, 1, 2, 2], 1))
print("lệch -1:    ", wrong_minus_one([1, 1, 2, 2], 1))
try:
    wrong_no_bound([10, 100, 101, 101], 101)
except IndexError as e:
    print("quên j < n: IndexError:", e)

đúng:        False
dùng >=:     True
lệch -1:     True
quên j < n: IndexError: list index out of range


## 13. Mẹo nhận dạng pattern

- Đề có chữ **sorted** + hỏi về **số lần xuất hiện / vị trí** → binary search (`bisect_left`, `bisect_right`).
- "Đếm số lần `x` trong mảng sắp xếp" = `bisect_right(x) - bisect_left(x)`.
- "Phần tử chiếm quá nửa" trong mảng **sắp xếp** luôn nằm ở **giữa**: `nums[n // 2]` (vì khối dài hơn nửa phải phủ index giữa).
- Mảng **không** sắp xếp, tìm majority → thuật toán **Boyer–Moore voting** (bài 169).

Bài liên quan: 34 (Find First and Last Position), 169 (Majority Element), 35 (Search Insert Position).

## 14. Biến thể

### 14.1. Phần tử giữa luôn là ứng viên duy nhất
Nếu mảng sắp xếp có majority element thì nó **phải** là `nums[n // 2]`. Do đó có thể kiểm tra nhanh `nums[n // 2] != target → False` trước.

In [21]:
def is_majority_mid_check(nums: List[int], target: int) -> bool:
    n = len(nums)
    if nums[n // 2] != target:
        return False                    # majority (nếu có) phải nằm ở giữa
    return bisect_right(nums, target) - bisect_left(nums, target) > n // 2

for nums, t, expected in cases:
    assert is_majority_mid_check(nums, t) == expected
print("Biến thể kiểm tra phần tử giữa: PASS")

Biến thể kiểm tra phần tử giữa: PASS


### 14.2. Mảng chưa sắp xếp — Boyer–Moore voting
Giữ một "ứng viên" và bộ đếm: gặp cùng ứng viên `+1`, khác `-1`, về 0 thì đổi ứng viên. Cuối cùng kiểm tra lại ứng viên bằng cách đếm.

In [22]:
def majority_boyer_moore(nums: List[int]):
    candidate, votes = None, 0
    for x in nums:
        if votes == 0:
            candidate = x
        votes += 1 if x == candidate else -1
    return candidate if nums.count(candidate) > len(nums) // 2 else None

print(majority_boyer_moore([3, 1, 3, 2, 3, 3]))   # 3
print(majority_boyer_moore([1, 2, 3, 1]))         # None

3
None


### 14.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Dùng khi |
|---|---|---|---|---|
| Đếm tuyến tính | duyệt hết | `O(n)` | `O(1)` | nhanh để viết |
| Hai bisect | đầu/cuối khối | `O(log n)` | `O(1)` | cần biết số lần |
| Một bisect + nhảy nửa | `nums[first + n//2]` | `O(log n)` | `O(1)` | **gọn nhất** |
| Kiểm tra giữa + bisect | loại sớm | `O(log n)` | `O(1)` | thêm tối ưu hằng số |
| Boyer–Moore | bỏ phiếu | `O(n)` | `O(1)` | mảng chưa sắp xếp |

## 15. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
Với `nums = [1, 3, 3, 3, 3, 4, 7]`, target = 3: `n // 2 = ?`, `first = ?`, `j = ?`, kết quả?

### Bài 2 — Tự viết `upper_bound`
Index đầu tiên `i` có `nums[i] > target` (giống `bisect_right`).

In [23]:
def upper_bound_exercise(nums: List[int], target: int) -> int:
    # TODO: giống lower_bound nhưng điều kiện đi phải là nums[mid] <= target
    pass

# assert upper_bound_exercise([2, 4, 5, 5, 5, 6], 5) == 5
# assert upper_bound_exercise([2, 4, 5, 5, 5, 6], 9) == 6
# assert upper_bound_exercise([2, 4, 5, 5, 5, 6], 0) == 0

### Bài 3 — Đếm số lần xuất hiện trong mảng sắp xếp bằng binary search tự viết

In [24]:
def count_sorted_exercise(nums: List[int], target: int) -> int:
    # TODO: dùng lower_bound và upper_bound_exercise
    pass

# assert count_sorted_exercise([1, 2, 2, 2, 3], 2) == 3
# assert count_sorted_exercise([1, 2, 2, 2, 3], 4) == 0

### Bài 4 — Vị trí đầu và cuối (LeetCode 34)
Trả `[first, last]` của `target`, hoặc `[-1, -1]` nếu không có.

In [25]:
def search_range_exercise(nums: List[int], target: int) -> List[int]:
    # TODO
    pass

# assert search_range_exercise([5, 7, 7, 8, 8, 10], 8) == [3, 4]
# assert search_range_exercise([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
# assert search_range_exercise([], 0) == [-1, -1]

### Bài 5 — Tìm majority element của mảng sắp xếp (không cho target)
Trả về majority element nếu có, ngược lại `None`.

In [26]:
def find_sorted_majority_exercise(nums: List[int]):
    # TODO: ứng viên duy nhất là nums[n // 2]
    pass

# assert find_sorted_majority_exercise([1, 2, 2, 2, 3]) == 2
# assert find_sorted_majority_exercise([1, 1, 2, 2]) is None

### Bài 6 — Mini challenge: phần tử xuất hiện hơn 25%
Mảng sắp xếp, đảm bảo có **đúng một** phần tử xuất hiện **hơn 25%** số lần (LeetCode 1287). Tìm nó trong `O(log n)` (chỉ cần thử vài ứng viên).

In [27]:
def over_quarter_exercise(nums: List[int]) -> int:
    # TODO: ứng viên nằm ở các vị trí n//4, n//2, 3n//4
    pass

# assert over_quarter_exercise([1, 2, 2, 6, 6, 6, 6, 7, 10]) == 6
# assert over_quarter_exercise([1, 1]) == 1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | dùng công thức `j = first + n//2` | `n = 7` | `first = 1`, `j = 4` |
| 2 | đổi `<` thành `<=` | `lo, hi = 0, n` | `if nums[mid] <= target: lo = mid + 1 else: hi = mid` |
| 3 | count = upper − lower | gọi 2 hàm | `return upper_bound(...) - lower_bound(...)` |
| 4 | khối liền nhau | kiểm tra `first < n and nums[first] == target` | last = upper − 1 |
| 5 | phần tử giữa | lấy `c = nums[n//2]` | đếm `c` bằng bisect, so `n // 2` |
| 6 | khối > n/4 phải phủ 1 trong 3 mốc | thử `nums[n//4]`, `nums[n//2]`, `nums[3n//4]` | đếm bằng bisect, trả ứng viên có count `> n // 4` |

## 17. Đáp án tham khảo

**Bài 1:** `n = 7`, `n // 2 = 3`, `first = 1`, `j = 4`, `nums[4] = 3` → **True** (3 xuất hiện 4 lần > 3.5).

In [28]:
def upper_bound_answer(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] <= target:
            lo = mid + 1
        else:
            hi = mid
    return lo

def count_sorted_answer(nums: List[int], target: int) -> int:
    return upper_bound_answer(nums, target) - lower_bound(nums, target)

def search_range_answer(nums: List[int], target: int) -> List[int]:
    first = lower_bound(nums, target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    return [first, upper_bound_answer(nums, target) - 1]

def find_sorted_majority_answer(nums: List[int]):
    n = len(nums)
    c = nums[n // 2]
    return c if count_sorted_answer(nums, c) > n // 2 else None

def over_quarter_answer(nums: List[int]) -> int:
    n = len(nums)
    for idx in (n // 4, n // 2, 3 * n // 4):
        c = nums[idx]
        if count_sorted_answer(nums, c) > n // 4:
            return c
    return nums[0]

trace_is_majority([1, 3, 3, 3, 3, 4, 7], 3)
assert upper_bound_answer([2, 4, 5, 5, 5, 6], 5) == 5
assert upper_bound_answer([2, 4, 5, 5, 5, 6], 9) == 6
assert upper_bound_answer([2, 4, 5, 5, 5, 6], 0) == 0
assert count_sorted_answer([1, 2, 2, 2, 3], 2) == 3
assert count_sorted_answer([1, 2, 2, 2, 3], 4) == 0
assert search_range_answer([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range_answer([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range_answer([], 0) == [-1, -1]
assert find_sorted_majority_answer([1, 2, 2, 2, 3]) == 2
assert find_sorted_majority_answer([1, 1, 2, 2]) is None
assert over_quarter_answer([1, 2, 2, 6, 6, 6, 6, 7, 10]) == 6
assert over_quarter_answer([1, 1]) == 1
print("Đáp án tham khảo PASS toàn bộ test!")

nums=[1, 3, 3, 3, 3, 4, 7], target=3, n=7
  vòng 1: lo=0, hi=7, mid=3, nums[mid]=3 >= 3 -> hi = 3
  vòng 2: lo=0, hi=3, mid=1, nums[mid]=3 >= 3 -> hi = 1
  vòng 3: lo=0, hi=1, mid=0, nums[mid]=1 < 3 -> lo = 1
  first = 1, j = first + n//2 = 4
  nums[4] = 3 -> == target -> True
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
from bisect import bisect_left

class Solution:
    def isMajorityElement(self, nums: List[int], target: int) -> bool:
        n = len(nums)
        j = bisect_left(nums, target) + n // 2
        return j < n and nums[j] == target
```

**Template binary search nửa mở:**
```python
lo, hi = 0, len(a)
while lo < hi:
    mid = (lo + hi) // 2
    if a[mid] < x: lo = mid + 1
    else:          hi = mid
return lo
```

**Checklist:**
- [ ] Majority là **quá** nửa: dùng `> n // 2` (hoặc `count * 2 > n`).
- [ ] Đã kiểm tra `j < n` trước khi truy cập `nums[j]`?
- [ ] Test ca **đúng một nửa** (`[1,1,2,2]`) → False?
- [ ] Test target vắng mặt, nhỏ/lớn hơn mọi phần tử?
- [ ] Nhớ `from bisect import bisect_left` và `from typing import List` khi chạy ngoài LeetCode.